# ERK-KTR Full FOV Stimulation Pipeline
This notebook showcases how to use the ERK-KTR full FOV stimulation pipeline. The pipeline is designed to simulate the full field of view (FOV) stimulation of a cells with the ERK-KTR biosensor. As it is a demo experiment, the pipeline runs on the demo hardware provided by MicroManager. 

### Import required libraries

In [1]:
import os
import time
from rtm_pymmcore.data_structures import Fov, Channel, StimTreatment
from pprint import pprint
import pandas as pd
import numpy as np
import dataclasses
import random

### Experimental Settings

In [2]:
# from rtm_pymmcore.microscope.MMDemo import MMDemo
# mic = MMDemo()
# mic.mmc.setChannelGroup("Channel")

# for Jungfrau Microscope enable here:
from rtm_pymmcore.microscope.Jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup("TTL_ERK")

In [ ]:
## Configuration options
FIRST_FRAME_STIMULATION = 10
# N_FRAMES = 340
N_FRAMES = 4


SLEEP_BEFORE_EXPERIMENT_START_in_H = 0

TIME_BETWEEN_TIMESTEPS = 5  # time in seconds between frames
TIME_PER_FOV = 3.3  # time in seconds per fov

MIX_STIM_EXPOSURE = False  # if True, the stim_exposure list will be randomized
ADD_STIM_EXPOSURE_GROUP = (
    False  # add an entry showing the last stimulation exposure for each FOV
)
REGULAR_SPACING_BETWEEN_STIMULATIONS = (
    False  # if True, the stim_timesteps will be evenly spaced
)

## Storage path for the experiment
base_path = "E:\\Cedric"
experiment_name = "2025-09-17_300min_Sustained"
path = os.path.join(base_path, experiment_name)

# Define Channels for which Images are taken. If no power is defined, the default power of the device will be used,
# for example, see the second channel "Cy5" below. The default power is set in the GUI
# Define Channels for which Images are taken. If no power is defined, the default power of the device will be used,
# for example, see the second channel "Cy5" below. The default power is set in the GUI
channels = []
channels.append(Channel(name="miRFP", exposure=300))
channels.append(Channel(name="mScarlet3", exposure=200))

# Channel to check for the expression of the optogenetic marker, can be used if it the marker is in the same channel as the stimulation channel.
channel_optocheck = Channel(name="mCitrine", exposure=300)
optocheck_timepoints = (0, 1)  # timepoints in frames when optocheck is done


# Condition mapping to FOVs. This is used to create a dataframe with the conditions and the FOVs.

# condition = [
#     "FGFR",
#     "EGFR",
#     "TrkA",
# ]
# condition = [
#     cond for cond in condition for _ in range(6)
# ] * 2  # means that we have 6 FOV per well, and 3 wells, so first 6 FOVs with FGFR,
# # then 6 with EGFR, then 6 with TrKA, and then again 6 with FGFR, etc.


condition = ["test"]

# condition = ["optoFGFR_high"] * 24 + ["optoFGFR"] * 24 # Example of adding multiple conditions to the dataframe. n repreats the amount of times the condition is repeated.


n_fovs_per_well = None  ## change this variable to the amount of fovs that you have per well. Set to None if you are not working with wellplate.


# Stimulation parameters for optogenetics. The stimulation will be repeated for each condition.

stim_exposures_timesteps = [
    {
        "ramp_pattern_name": "Sustained",
        "stim_exposure_list": (250,) * 300,
        "stim_timestep": range(10, 311, 1),
    }
]

stim_exposures_timesteps = [
    {
        "ramp_pattern_name": "Sustained",
        "stim_exposure_list": (250,) * 3,
        "stim_timestep": (0, 1, 2),
    }
]


for stim_exposure_timestep in stim_exposures_timesteps:
    if isinstance(stim_exposure_timestep["stim_timestep"], range):
        stim_exposure_timestep["stim_timestep"] = tuple(
            stim_exposure_timestep["stim_timestep"]
        )
    if isinstance(stim_exposure_timestep["stim_exposure_list"], range):
        stim_exposure_timestep["stim_exposure_list"] = tuple(
            stim_exposure_timestep["stim_exposure_list"]
        )
    if isinstance(stim_exposure_timestep["stim_timestep"], list):
        stim_exposure_timestep["stim_timestep"] = tuple(
            stim_exposure_timestep["stim_timestep"]
        )
    if isinstance(stim_exposure_timestep["stim_exposure_list"], list):
        stim_exposure_timestep["stim_exposure_list"] = tuple(
            stim_exposure_timestep["stim_exposure_list"]
        )


if MIX_STIM_EXPOSURE:
    for stim_exposure_timestep in stim_exposures_timesteps:
        stim_exp_list = list(stim_exposure_timestep["stim_exposure_list"])
        random.shuffle(stim_exp_list)
        stim_exposure_timestep["stim_exposure_list"] = tuple(stim_exp_list)

for stexptim in stim_exposures_timesteps:
    print("Pattern Name: ", stexptim["ramp_pattern_name"])

    for stim_exp, stim_timestep in zip(
        stexptim["stim_exposure_list"], stexptim["stim_timestep"]
    ):
        print(f"{stim_exp} at {stim_timestep}")
    print("")

    # add general stimulation parameters
    stexptim["stim_power"] = 10
    stexptim["stim_channel_name"] = "CyanStim"
    stexptim["stim_channel_group"] = "TTL_ERK"
    stexptim["stim_channel_device_name"] = "Spectra"
    stexptim["stim_channel_power_property_name"] = "Cyan_Level"


## Define the Tools that you are using for the experiment
from rtm_pymmcore.stimulation.base_stimulation import StimWholeFOV
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.erk_ktr import FE_ErkKtr
from rtm_pymmcore.feature_extraction.optocheck_fe import OptoCheckFE
from rtm_pymmcore.segmentation.cellpose_v4 import CellposeV4

segmentators = [
    {
        "name": "labels",
        "class": CellposeV4(
            custom_model_path="E:\\models\\cellpose\\LifeActH2B_mixed_with_only_H2B_v1",
            min_size=100,
        ),
        "use_channel": 0,
        "save_tracked": True,
    },
]

stimulator = StimWholeFOV()
feature_extractor = FE_ErkKtr("labels")
tracker = TrackerTrackpy(search_range=25)
optocheck = OptoCheckFE("labels", multi_timepoint=True)

from rtm_pymmcore.img_processing_pip import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_optocheck=optocheck,
)
mic.set_pipeline(pipeline=pipeline)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Pattern Name:  Sustained
250 at 0
250 at 1
250 at 2

Directory E:\Cedric\2025-09-17_300min_Sustained\raw already exists
Directory E:\Cedric\2025-09-17_300min_Sustained\tracks already exists
Directory E:\Cedric\2025-09-17_300min_Sustained\stim_mask already exists
Directory E:\Cedric\2025-09-17_300min_Sustained\stim already exists
Directory E:\Cedric\2025-09-17_300min_Sustained\particles already exists
Directory E:\Cedric\2025-09-17_300min_Sustained\labels_ring already exists
Directory E:\Cedric\2025-09-17_300min_Sustained\labels already exists
Directory E:\Cedric\2025-09-17_300min_Sustained\optocheck already exists


### GUI - Napari Micromanager

#### Load GUI

In [4]:
### Base GUI ###
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None
load_from_file = False

c:\Users\Jungfrau\Documents\alandolt\code\rtm-pymmcore\.venv\Lib\site-packages\napari_micromanager\main_window.py:51: FutureWarning: The `_dock_widgets` property is private and should not be used in any plugin code. Please use the `dock_widgets` property instead.
  if "MinMax" not in getattr(self.viewer.window, "_dock_widgets", []):


Please execute the following code block, if you would like to set a custom ROI (e.g. smaller illumination than the full field of view of the camera). Execute it after you have started the camera once in the GUI. 

In [ ]:
if mic.SET_ROI_REQUIRED:
    mic.mmc.clearROI()
    mic.mmc.setROI(mic.ROI_X, mic.ROI_Y, mic.ROI_WIDTH, mic.ROI_HEIGHT)

### Map Experiment to FOVs

#### If FOVs already saved - Reload them from file

In [4]:
import json

# file = os.path.join(path, "fovs.json")
file = os.path.join("fovs.json")
with open(file, "r") as f:
    data_mda_fovs = json.load(f)
data_mda_fovs = data_mda_fovs
load_from_file = True

### Use FOVs to generate dataframe for acquisition

In [19]:
n_fovs_simultaneously = TIME_BETWEEN_TIMESTEPS // TIME_PER_FOV
timesteps = range(N_FRAMES)


start_time = 0
if not load_from_file:
    data_mda_fovs = viewer.window._dock_widgets["MDA"].widget().value().stage_positions
    data_mda_fovs_dict = []
    for data_mda in data_mda_fovs:
        data_mda_fovs_dict.append(data_mda.model_dump())
    data_mda_fovs = data_mda_fovs_dict
    if data_mda_fovs is None:
        assert False, "No fovs selected. Please select fovs in the MDA widget"

if "channel_optocheck" not in locals():
    channel_optocheck = None
if (
    "optocheck_timepoints" not in locals() and channel_optocheck is not None
) or optocheck_timepoints is None:
    optocheck_timepoints = [N_FRAMES - 1]
dfs = []
fovs = []
for fov_index, fov in enumerate(data_mda_fovs):
    fov_object = Fov(fov_index)
    fovs.append(fov_object)
    fov_group = fov_index // n_fovs_simultaneously
    start_time = fov_group * TIME_BETWEEN_TIMESTEPS * len(timesteps)
    if len(condition) == 1:
        condition_fov = condition[0]
    else:
        condition_fov = condition[fov_index]
    for timestep in timesteps:
        row = {
            "fov_object": fov_object,
            "fov": fov_index,
            "fov_x": fov.get("x"),
            "fov_y": fov.get("y"),
            "fov_z": fov.get("z") if not mic.USE_ONLY_PFS else None,
            "fov_name": str(fov_index) if fov["name"] is None else fov["name"],
            "timestep": timestep,
            "time": start_time + timestep * TIME_BETWEEN_TIMESTEPS,
            "cell_line": condition_fov,
            "channels": tuple(dataclasses.asdict(channel) for channel in channels),
            "fname": f"{str(fov_index).zfill(3)}_{str(timestep).zfill(5)}",
        }
        if channel_optocheck is not None:
            row["optocheck"] = True if timestep in optocheck_timepoints else False

            if isinstance(channel_optocheck, list):
                row["optocheck_channels"] = tuple(
                    dataclasses.asdict(channel) for channel in channel_optocheck
                )
            else:
                row["optocheck_channels"] = tuple(
                    [dataclasses.asdict(channel_optocheck)]
                )

        dfs.append(row)

df_acquire = pd.DataFrame(dfs)

print(f"Total Experiment Time: {df_acquire['time'].max()/3600}h")


n_fovs = len(data_mda_fovs)
n_stim_treatments = len(stim_exposures_timesteps)
if n_stim_treatments > 0:
    n_fovs_per_stim_condition = n_fovs // n_stim_treatments // len(np.unique(condition))
    stim_treatment_tot = []
    random.shuffle(stim_exposures_timesteps)
    if n_fovs_per_well is None:
        for fov_index in range(0, n_fovs_per_stim_condition):
            stim_treatment_tot.extend(stim_exposures_timesteps)
        random.shuffle(stim_treatment_tot)

        if n_fovs % n_stim_treatments != 0:
            print(
                f"Warning: Not equal number of fovs per stim condition. {n_fovs % n_stim_treatments} fovs will have repeated treatment"
            )
            stim_treatment_tot.extend(
                stim_exposures_timesteps[: n_fovs % n_stim_treatments]
            )
        print(f"Doing {n_fovs_per_stim_condition} experiment per stim condition")

        if len(condition) != 1:
            stim_treatment_tot = stim_treatment_tot * len(np.unique(condition))

        df_acquire = pd.merge(
            df_acquire,
            pd.DataFrame(stim_treatment_tot),
            left_on="fov",
            right_index=True,
        )
    else:
        stim_treatment_tot = []
        for cell_line in np.unique(condition):
            fovs_for_one_cell_line = df_acquire.query(f"cell_line == @cell_line")[
                "fov"
            ].unique()
            stim_treat = [
                stim
                for stim in stim_exposures_timesteps
                for _ in range(n_fovs_per_well)
            ]
            if len(fovs_for_one_cell_line) != len(stim_treat):
                print(
                    f"Warning: Number of fovs ({len(fovs_for_one_cell_line)}) for cell line {cell_line} does not match number of stim treatments ({len(stim_treat)})."
                )
            stim_treat = pd.DataFrame(stim_treat)
            stim_treat["fov"] = fovs_for_one_cell_line
            stim_treatment_tot.append(stim_treat)
        stim_treat = pd.concat(stim_treatment_tot, ignore_index=True)
        df_acquire = pd.merge(
            df_acquire, stim_treat, left_on="fov", right_on="fov", how="left"
        )

    df_acquire["stim_exposure"] = np.nan

    for fov in df_acquire["fov"].unique():
        fov_data = df_acquire[df_acquire["fov"] == fov]

        stim_pattern = fov_data.iloc[0]

        if isinstance(stim_pattern["stim_timestep"], tuple) and isinstance(
            stim_pattern["stim_exposure_list"], tuple
        ):
            exposure_map = dict(
                zip(stim_pattern["stim_timestep"], stim_pattern["stim_exposure_list"])
            )

            for timestep in fov_data["timestep"]:
                if timestep in exposure_map:
                    mask = (df_acquire["fov"] == fov) & (
                        df_acquire["timestep"] == timestep
                    )
                    df_acquire.loc[mask, "stim_exposure"] = exposure_map[timestep]

    df_acquire["stim"] = df_acquire.apply(
        lambda row: (
            row["timestep"] in row["stim_timestep"] and row["stim_exposure"] > 0
        ),
        axis=1,
    )


pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", True)
df_acquire = df_acquire.sort_values(by=["time", "fov"]).reset_index(drop=True)
df_acquire = df_acquire.dropna(axis=1, how="all")
if ADD_STIM_EXPOSURE_GROUP and REGULAR_SPACING_BETWEEN_STIMULATIONS:
    spacing_interval = (
        df_acquire["stim_timestep"][0][1] - df_acquire["stim_timestep"][0][0]
    )
    for start in range(0, df_acquire["timestep"].max(), spacing_interval):
        end = start + spacing_interval
        mask = (df_acquire["timestep"] >= start) & (df_acquire["timestep"] < end)
        window = df_acquire.loc[mask, "stim_exposure"]
        value = window.dropna().iloc[0] if window.dropna().size > 0 else np.nan
        df_acquire.loc[mask, "stim_exposure"] = value

else:
    df_acquire["stim_exposure"] = df_acquire["stim_exposure"].fillna(0)
df_acquire[
    [
        "fov",
        "timestep",
        "time",
        "cell_line",
        "stim_power",
        "stim_exposure",
        "stim_exposure_list",
        "stim_timestep",
        "stim",
        "ramp_pattern_name",
        "optocheck",
    ]
]

Total Experiment Time: 0.004166666666666667h
Doing 1 experiment per stim condition


,fov,timestep,time,cell_line,stim_power,stim_exposure,stim_exposure_list,stim_timestep,stim,ramp_pattern_name,optocheck
0,0,0,0.0,test,10,250.0,"(250, 250, 250)","(0, 1, 2)",True,Sustained,True
1,0,1,5.0,test,10,250.0,"(250, 250, 250)","(0, 1, 2)",True,Sustained,True
2,0,2,10.0,test,10,250.0,"(250, 250, 250)","(0, 1, 2)",True,Sustained,False
3,0,3,15.0,test,10,0.0,"(250, 250, 250)","(0, 1, 2)",False,Sustained,False


### Run experiment

In [20]:
for _ in range(0, SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600):
    time.sleep(1)

try:

    mm_wdg._core_link.cleanup()

except:
    pass


mic.run_experiment(df_acquire)
print("Experiment finished")
mic.post_experiment()

time.sleep(10)

fovs_i_list = os.listdir(os.path.join(path, "tracks"))
fovs_i_list.sort()
dfs = []

for fov_i in fovs_i_list:

    track_file = os.path.join(path, "tracks", fov_i)
    df = pd.read_parquet(track_file)
    dfs.append(df)

pd.concat(dfs).to_parquet(os.path.join(path, "exp_data.parquet"))

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

Experiment finished


### Function to re-connect link with GUI if manually broken

The following functions can be used to manually re-make the connection between the GUI and the running rtm-pymmcore script. However, normally you don't need to execute them. 

In [ ]:
### Manually reconnect pymmcore with napari-micromanager
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

The following code block can be used to manually break the connection between GUI and Jupyter Notebook:


In [ ]:
### Break connection
# mm_wdg._core_link.cleanup()